In [0]:
import json
from collections import Counter

land_dir = "/Volumes/laddstolpar_df/landing/raw/nobil/snapshot_date=2026-09-25"
with open(f"{land_dir}/nobil_swe.json", encoding="utf-8") as f:
    stations = json.load(f)["chargerstations"]
with open(f"{land_dir}/stats_totals_all_counties.json", encoding="utf-8") as f:
    stats = {c["countyid"]: (c["county"], c["count"]) for c in json.load(f)["chargerstations"]}
csmd = [s["csmd"] for s in stations]

# 1. Freshness: is the dump a stale cached export?
print("max Created:", max(c["Created"] for c in csmd if c.get("Created")))
print("max Updated:", max(c["Updated"] for c in csmd if c.get("Updated")))

# 2. Where are the missing 458? Per county, largest gap first
dump_by_county = Counter(c.get("County_ID") for c in csmd)
rows = [(cid, name, n, dump_by_county.get(cid, 0)) for cid, (name, n) in stats.items()]
for cid, name, n, d in sorted(rows, key=lambda r: r[3] - r[2]):
    print(f"{cid} {name:<16} stats {n:>5}  dump {d:>5}  diff {d - n:+}")
print("County_ID in dump but not in stats:", set(dump_by_county) - set(stats))

# 3. Real-time attribute (st 21). PHP encodes an empty object as [], so guard for lists
def st_of(s):
    st = (s.get("attr") or {}).get("st") or {}
    return st if isinstance(st, dict) else {}
print("attr.st is a list:", sum(isinstance((s.get("attr") or {}).get("st"), list) for s in stations))
print("Real-time information:", Counter(st_of(s).get("21", {}).get("trans", "<missing>") for s in stations))

In [0]:
import os, time, requests
from collections import Counter
from datetime import datetime
from zoneinfo import ZoneInfo

NOBIL_KEY = dbutils.secrets.get(scope="kv-labb2", key="nobil-api-key")
SEARCH_URL = "https://nobil.no/api/server/search.php"

DUMP_URL = "https://nobil.no/api/server/datadump.php"

def nobil_post(url, timeout=300, **params):
    # POST keeps the key out of the URL (and out of exception messages)
    resp = requests.post(url, data={"apikey": NOBIL_KEY, **params}, timeout=timeout)
    resp.raise_for_status()
    return resp

snapshot_date = datetime.now(ZoneInfo("Europe/Stockholm")).date().isoformat()
land_dir   = f"/Volumes/laddstolpar_df/landing/raw/nobil/snapshot_date={snapshot_date}"
stats_path = f"{land_dir}/stats_totals_all_counties.json"
dump_path  = f"{land_dir}/nobil_swe.json"          # written last = completion marker


In [0]:
# 4. Same dump with both filters explicitly off, kept in memory, not landed
test = nobil_post(DUMP_URL, countrycode="SWE", format="json", file="false",
                  norealtime="false", nonimupdate="false").json()["chargerstations"]
landed_ids = {c["id"] for c in csmd}
new = [s["csmd"] for s in test if s["csmd"]["id"] not in landed_ids]
print(f"{len(test):,} stations, {len(new):,} not in the landed file")
print("Owners of the new ones:", Counter(c.get("Owned_by") for c in new).most_common(8))

cell 5 

In [0]:
odd = [c for c in csmd if c.get("County_ID") in {"1083", "1084", "1085"}]
print(len(odd), "stations")
print(Counter((c["County_ID"], c.get("County"), c.get("Municipality_ID"), c.get("Municipality"))
              for c in odd))

In [0]:
r = nobil_post(SEARCH_URL, timeout=120, apiversion="3", action="search", format="json",
               type="rectangle", northeast="(58.30, 15.70)", southwest="(56.85, 13.10)",
               limit="5000")
found   = [s["csmd"] for s in r.json()["chargerstations"]]
found06 = [c for c in found if c.get("County_ID") == "06"]
missing = [c for c in found06 if c["id"] not in landed_ids]

print(f"rectangle {len(found):,} | county 06 {len(found06)} (stats 418, dump 358) | missing {len(missing)}")
print("Station_status, county 06:", Counter(c.get("Station_status") for c in found06))
print("Station_status, missing:  ", Counter(c.get("Station_status") for c in missing))
if missing:
    print("Fields only in missing records:", set(missing[0]) - set(csmd[0]))
    for c in missing[:5]:
        print(c["id"], c.get("name"), "|", c.get("Municipality"), "| status", c.get("Station_status"),
              "| created", c.get("Created"), "|", c.get("Owned_by"))

In [0]:
# det = nobil_search(type="stats_DetailedTotalsByCountyId", id="06", countrycode="SWE")
det = nobil_post(SEARCH_URL, timeout=60, apiversion="3", action="search", format="json",
                 type="stats_DetailedTotalsByCountyId", id="06", countrycode="SWE").json()
rows = det["chargerstations"]
print("keys:", list(rows[0]))

id_key  = next(k for k in rows[0] if "munic" in k.lower() and "id" in k.lower())
dump06  = Counter(c.get("Municipality_ID") for c in csmd if c.get("County_ID") == "06")
for r in sorted(rows, key=lambda r: dump06.get(r[id_key], 0) - r["count"]):
    d = dump06.get(r[id_key], 0)
    print(f"{r[id_key]} {str(r.get('municipality', '')):<14} stats {r['count']:>4}  dump {d:>4}  diff {d - r['count']:+}")

In [0]:
%sql
DESCRIBE HISTORY laddstolpar_df.bronze.nobil_stations

In [0]:
%sql
SELECT c.value:['5'].attrvalid::string AS attrvalid,
       c.value:['5'].trans::string     AS trans,
       count(*)                         AS connectors,
       count_if(c.value:['5'].attrval::string NOT IN ('', '0')) AS attrval_filled
FROM laddstolpar_df.bronze.nobil_stations,
     LATERAL variant_explode(attr:conn) AS c
GROUP BY 1, 2
ORDER BY connectors DESC;

In [0]:
from pyspark.sql import functions as F
display(spark.sql("DESCRIBE HISTORY laddstolpar_df.bronze.nobil_stations")
        .select("version", "timestamp", "operation",
                F.col("operationMetrics")["numOutputRows"].alias("rows"),
                F.col("operationMetrics")["numFiles"].alias("files"),
                F.current_timestamp().alias("checked_at")))

In [0]:
%sql
WITH conn AS (
  SELECT s.station_id, c.value AS conn
  FROM laddstolpar_df.bronze.nobil_stations s,
       LATERAL variant_explode(s.attr:conn) AS c
),
cap AS (
  SELECT conn:['5'].attrvalid::string AS attrvalid, count(*) AS n_connectors
  FROM conn
  GROUP BY ALL
)
SELECT cap.attrvalid, cap.n_connectors,
       sd.kw, sd.current_type, sd.kw_basis, sd.nobil_label
FROM cap
LEFT JOIN laddstolpar_df.bronze.seed_nobil_capacity sd
  ON sd.attrvalid = cap.attrvalid
ORDER BY cap.n_connectors DESC;

In [0]:
%sql
WITH conn AS (
  SELECT c.value AS conn
  FROM laddstolpar_df.bronze.nobil_stations s,
       LATERAL variant_explode(s.attr:conn) AS c
)
SELECT conn:['5'].attrvalid::string AS attrvalid,
       conn:['5'].trans::string     AS trans,
       conn:['5'].attrval::string   AS attrval,
       count(*)                     AS n
FROM conn
WHERE conn:['5'].attrvalid::string IN ('43', '46', '48', '50')
GROUP BY ALL
ORDER BY attrvalid, n DESC;

In [0]:
%sql
-- connector attribute inventory
WITH conn AS (
  SELECT c.value AS conn
  FROM laddstolpar_df.bronze.nobil_stations s,
       LATERAL variant_explode(s.attr:conn) AS c
)
SELECT a.key                                                     AS attrtypeid,
       any_value(a.value:attrname::string)                       AS attrname,
       count(*)                                                  AS n_connectors,
       count_if(nullif(a.value:attrval::string, '') IS NOT NULL) AS n_attrval_filled,
       count(DISTINCT a.value:attrvalid::string)                 AS n_distinct_codes
FROM conn, LATERAL variant_explode(conn) AS a
GROUP BY a.key
ORDER BY n_connectors DESC;

In [0]:
%sql
-- query A: EVSE rule impact
WITH conn AS (
  SELECT s.station_id,
         s.csmd:Number_charging_points::bigint       AS n_points,
         c.key                                       AS conn_no,
         nullif(c.value:['28'].attrval::string, '')  AS evse_id,
         c.value:['5'].attrvalid::string             AS cap_code
  FROM laddstolpar_df.bronze.nobil_stations s,
       LATERAL variant_explode(s.attr:conn) AS c
),
k AS (
  SELECT conn.*, CAST(sd.kw AS DOUBLE) AS kw
  FROM conn
  JOIN laddstolpar_df.bronze.seed_nobil_capacity sd ON sd.attrvalid = conn.cap_code
  WHERE sd.current_type IN ('AC', 'DC')
),
evse AS (   -- one row per EVSE; a connector without an EVSE ID is its own EVSE
  SELECT station_id,
         max(n_points)                                   AS n_points,
         coalesce(evse_id, concat('conn#', conn_no))     AS evse_key,
         max(evse_id) IS NOT NULL                        AS has_id,
         count(*)                                        AS n_conn,
         max(kw)                                         AS kw_max,
         sum(kw)                                         AS kw_sum
  FROM k
  GROUP BY station_id, coalesce(evse_id, concat('conn#', conn_no))
),
st AS (
  SELECT station_id, max(n_points) AS n_points, count(*) AS n_evse, sum(n_conn) AS n_conn
  FROM evse
  GROUP BY station_id
)
SELECT
  (SELECT sum(n_conn) FROM evse)                               AS connectors,
  (SELECT count(*) FROM evse)                                  AS evse_groups,
  (SELECT count_if(has_id AND n_conn > 1) FROM evse)           AS shared_evse_groups,
  (SELECT sum(n_conn) FROM evse WHERE has_id AND n_conn > 1)   AS connectors_in_shared,
  (SELECT round(sum(kw_sum) / 1000, 1) FROM evse)              AS mw_naive,
  (SELECT round(sum(kw_max) / 1000, 1) FROM evse)              AS mw_evse_rule,
  count_if(n_evse = n_points)                                  AS st_evse_eq_points,
  count_if(n_evse > n_points)                                  AS st_evse_gt_points,
  count_if(n_evse < n_points)                                  AS st_evse_lt_points,
  count_if(n_conn = n_points)                                  AS st_conn_eq_points
FROM st;

In [0]:
%sql
-- query B: vehicle types
SELECT c.value:['17'].attrvalid::string              AS vehicle_type,
       any_value(c.value:['17'].trans::string)        AS label,
       count(*)                                       AS connectors,
       count_if(sd.current_type = 'DC')               AS dc_connectors,
       round(sum(try_cast(sd.kw AS DOUBLE)) / 1000, 1) AS mw_naive
FROM laddstolpar_df.bronze.nobil_stations s,
     LATERAL variant_explode(s.attr:conn) AS c
LEFT JOIN laddstolpar_df.bronze.seed_nobil_capacity sd
  ON sd.attrvalid = c.value:['5'].attrvalid::string
GROUP BY 1
ORDER BY connectors DESC;

In [0]:
%sql
SELECT current_type, kw_basis, count(*) AS codes, min(kw) AS min_kw, max(kw) AS max_kw
FROM laddstolpar_df.silver.nobil_capacity
GROUP BY ALL ORDER BY ALL;

In [0]:
%sql
-- nobil summary query
SELECT
  (SELECT count(*)                          FROM laddstolpar_df.silver.nobil_station)   AS stations,         -- expect 9,019
  (SELECT count_if(is_public)               FROM laddstolpar_df.silver.nobil_station)   AS public_stations,
  (SELECT count(DISTINCT kommun_kod)        FROM laddstolpar_df.silver.nobil_station)   AS kommuner,
  (SELECT count_if(station_name IS NULL)    FROM laddstolpar_df.silver.nobil_station)   AS no_name,          -- all NULL = wrong field name
  (SELECT count_if(operator IS NULL)        FROM laddstolpar_df.silver.nobil_station)   AS no_operator,
  (SELECT count_if(lat IS NULL)             FROM laddstolpar_df.silver.nobil_station)   AS no_position,      -- expect 0
  (SELECT count(*)                          FROM laddstolpar_df.silver.nobil_connector) AS connectors,       -- 60,270 minus the Danish stations' connectors
  (SELECT count_if(current_type IN ('AC', 'DC') AND kw IS NULL)
                                            FROM laddstolpar_df.silver.nobil_connector) AS charging_without_kw, -- expect 0
  (SELECT round(sum(kw) / 1000, 1)          FROM laddstolpar_df.silver.nobil_connector) AS mw_naive;         -- a little below 3,745.1